In [14]:
# ================================
# 📊 NEPSE MARKET DATA PIPELINE
# ================================

# Flow:
# Mero Lagani Website
#   ↓
# Playwright Browser Automation
#   ↓
# Extract Tables:
#   - Turnover
#   - Gainers
#   - Losers
#   ↓
# Clean Data (Python)
#   ↓
# Store in SQLite (nepse.db)
#   ↓
# Future: Analysis + Trading Signals

# Tools:
# - Playwright
# - Python
# - SQLite
# - Jupyter Notebook

# Status:
# ✔ Scraping working
# ✔ Database working
# ⏳ Automation next (GitHub Actions)

In [1]:
import sqlite3

conn = sqlite3.connect("nepse.db")

conn.execute("""
CREATE TABLE IF NOT EXISTS market_leaders (
    date TEXT,
    symbol TEXT,
    category TEXT,
    rank INTEGER,
    value TEXT
)
""")

conn.commit()
conn.close()

print("Database ready")

Database ready


In [3]:
import sqlite3
from datetime import date
from playwright.async_api import async_playwright

URL = "https://www.merolagani.com/MarketSummary.aspx"

async def run():
    today = str(date.today())

    conn = sqlite3.connect("nepse.db")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto(URL, wait_until="domcontentloaded")
        await page.wait_for_timeout(5000)

        tables = page.locator("table")

        # --- TABLE MAPPING (from your inspection) ---
        turnover_table = tables.nth(2)
        gainers_table = tables.nth(4)
        losers_table = tables.nth(5)

        # -------- TURNOVER --------
        rows = await turnover_table.locator("tr").all()
        for i, row in enumerate(rows[1:]):
            cols = await row.locator("td").all_text_contents()
            if len(cols) >= 2:
                conn.execute("""
                    INSERT INTO market_leaders
                    VALUES (?, ?, ?, ?, ?)
                """, (today, cols[0], "turnover", i+1, cols[1]))

        # -------- GAINERS --------
        rows = await gainers_table.locator("tr").all()
        for i, row in enumerate(rows[1:]):
            cols = await row.locator("td").all_text_contents()
            if len(cols) >= 3:
                conn.execute("""
                    INSERT INTO market_leaders
                    VALUES (?, ?, ?, ?, ?)
                """, (today, cols[0], "gainer", i+1, cols[2]))

        # -------- LOSERS --------
        rows = await losers_table.locator("tr").all()
        for i, row in enumerate(rows[1:]):
            cols = await row.locator("td").all_text_contents()
            if len(cols) >= 3:
                conn.execute("""
                    INSERT INTO market_leaders
                    VALUES (?, ?, ?, ?, ?)
                """, (today, cols[0], "loser", i+1, cols[2]))

        conn.commit()
        conn.close()

        await browser.close()

await run()

In [4]:
import sqlite3

conn = sqlite3.connect("nepse.db")

rows = conn.execute("SELECT * FROM market_leaders LIMIT 10").fetchall()

conn.close()

rows

[('2026-06-05', 'EBLD86', 'turnover', 1, '340,153,074.30'),
 ('2026-06-05', 'AKJCL', 'turnover', 2, '320,564,333.30'),
 ('2026-06-05', 'CFCL', 'turnover', 3, '185,265,984.40'),
 ('2026-06-05', 'NRIC', 'turnover', 4, '171,989,587.80'),
 ('2026-06-05', 'SAPDBL', 'turnover', 5, '144,706,018.70'),
 ('2026-06-05', 'RIDI', 'turnover', 6, '140,978,846.40'),
 ('2026-06-05', 'NICL', 'turnover', 7, '104,544,813.60'),
 ('2026-06-05', 'BHCL', 'turnover', 8, '84,618,321.70'),
 ('2026-06-05', 'API', 'turnover', 9, '83,071,719.80'),
 ('2026-06-05', 'NHPC', 'turnover', 10, '77,615,421.10')]

In [6]:
import sqlite3

conn = sqlite3.connect("nepse.db")

rows = conn.execute("""
SELECT date, symbol, category, rank, value
FROM market_leaders
ORDER BY date DESC
LIMIT 20
""").fetchall()

conn.close()

rows

[('2026-06-05', 'EBLD86', 'turnover', 1, '340,153,074.30'),
 ('2026-06-05', 'AKJCL', 'turnover', 2, '320,564,333.30'),
 ('2026-06-05', 'CFCL', 'turnover', 3, '185,265,984.40'),
 ('2026-06-05', 'NRIC', 'turnover', 4, '171,989,587.80'),
 ('2026-06-05', 'SAPDBL', 'turnover', 5, '144,706,018.70'),
 ('2026-06-05', 'RIDI', 'turnover', 6, '140,978,846.40'),
 ('2026-06-05', 'NICL', 'turnover', 7, '104,544,813.60'),
 ('2026-06-05', 'BHCL', 'turnover', 8, '84,618,321.70'),
 ('2026-06-05', 'API', 'turnover', 9, '83,071,719.80'),
 ('2026-06-05', 'NHPC', 'turnover', 10, '77,615,421.10'),
 ('2026-06-05', 'KHPL', 'gainer', 1, '14.99'),
 ('2026-06-05', 'NYADI', 'gainer', 2, '9.16'),
 ('2026-06-05', 'SBIBD86', 'gainer', 3, '7.21'),
 ('2026-06-05', 'NICD88', 'gainer', 4, '5.8'),
 ('2026-06-05', 'NBLD85', 'gainer', 5, '5.77'),
 ('2026-06-05', 'EBLD86', 'gainer', 6, '5.42'),
 ('2026-06-05', 'NMBD87/88', 'gainer', 7, '5.31'),
 ('2026-06-05', 'CFCL', 'gainer', 8, '4.78'),
 ('2026-06-05', 'PBD85', 'gainer', 

In [7]:
import sqlite3

conn = sqlite3.connect("nepse.db")

gainers = conn.execute("""
SELECT * FROM market_leaders
WHERE category = 'gainer'
LIMIT 10
""").fetchall()

losers = conn.execute("""
SELECT * FROM market_leaders
WHERE category = 'loser'
LIMIT 10
""").fetchall()

conn.close()

gainers, losers

([('2026-06-05', 'KHPL', 'gainer', 1, '14.99'),
  ('2026-06-05', 'NYADI', 'gainer', 2, '9.16'),
  ('2026-06-05', 'SBIBD86', 'gainer', 3, '7.21'),
  ('2026-06-05', 'NICD88', 'gainer', 4, '5.8'),
  ('2026-06-05', 'NBLD85', 'gainer', 5, '5.77'),
  ('2026-06-05', 'EBLD86', 'gainer', 6, '5.42'),
  ('2026-06-05', 'NMBD87/88', 'gainer', 7, '5.31'),
  ('2026-06-05', 'CFCL', 'gainer', 8, '4.78'),
  ('2026-06-05', 'PBD85', 'gainer', 9, '4.7'),
  ('2026-06-05', 'BOKD86', 'gainer', 10, '4.2')],
 [('2026-06-05', 'VLBS', 'loser', 1, '-4.69'),
  ('2026-06-05', 'BGWT', 'loser', 2, '-4.27'),
  ('2026-06-05', 'HFIN', 'loser', 3, '-3.66'),
  ('2026-06-05', 'AKJCL', 'loser', 4, '-3.43'),
  ('2026-06-05', 'KKHC', 'loser', 5, '-3.26'),
  ('2026-06-05', 'NMIC', 'loser', 6, '-3.13'),
  ('2026-06-05', 'KBSH', 'loser', 7, '-3'),
  ('2026-06-05', 'LVF2', 'loser', 8, '-2.82'),
  ('2026-06-05', 'HEI', 'loser', 9, '-2.73'),
  ('2026-06-05', 'GRDBL', 'loser', 10, '-2.71')])